# 01 - Data Exploration

This notebook reads the raw loan dataset from S3 and profiles the source data before any transformation.

In [ ]:
from pyspark.sql.functions import col, count, isnan, when

RAW_FILE_PATH = "s3://loanshield-raw/accepted_2007_to_2018Q4.csv"
SAMPLE_ROWS = 10000  # Use None for final full-data portfolio run

df_raw = spark.read.csv(RAW_FILE_PATH, header=True, inferSchema=True)
if SAMPLE_ROWS:
    df_raw = df_raw.limit(SAMPLE_ROWS)

print(f"Raw records: {df_raw.count():,}")
print(f"Raw columns: {len(df_raw.columns)}")

In [ ]:
df_raw.printSchema()

In [ ]:
df_raw.limit(10).show(truncate=False)

In [ ]:
key_columns = ["id", "loan_amnt", "term", "int_rate", "grade", "annual_inc", "loan_status", "dti", "fico_range_low"]
available_key_columns = [c for c in key_columns if c in df_raw.columns]

null_checks = [
    count(when(col(c).isNull(), c)).alias(c)
    for c in available_key_columns
]
df_raw.select(null_checks).show(truncate=False)

In [ ]:
df_raw.groupBy("loan_status").count().orderBy("count", ascending=False).show(truncate=False)
df_raw.groupBy("grade").count().orderBy("grade").show(truncate=False)